# 📘 Colab Notebook: Evaluate LLM Responses from AWS S3 Using Llumo

## 📝 Notebook Overview
This notebook demonstrates how to load your existing LLM evaluation data from an AWS S3 bucket, format it, and then evaluate it using Llumo’s powerful input-level metrics to ensure quality and safety.

### ✨ Metrics included:

- 🎯 Response Correctness
- 🧩 Response Completeness
- 🧠 Response Bias
- ☣️ Response Harmfulness
- ▶ Hallucination
- 🛠️ Context Utilization
  
---

## 🚀 What you will do in this notebook:
- ☁️ Connect to AWS S3 and load a dataset of queries, contexts, and model outputs.  
- 🔄 Format the raw data into the standardized structure required by Llumo.
- 🤖 Evaluate the responses for correctness, completeness, bias, harmfulness, and more.
- 📊 View the detailed evaluation results in a structured table.  
---

### **⚙️ 1. Install Dependencies**
First, we'll install the necessary Python libraries. `llumo` is the official SDK for the Llumo platform, `boto3` is the AWS SDK for Python to interact with S3, and `pandas` is used for data manipulation and display.

In [ ]:
!pip install llumo boto3 pandas -q

### **📚 2. Import Required Libraries**

In [ ]:
import os
import boto3
import pandas as pd
import json
from llumo import LlumoClient
import getpass

### **🔑 3. Configure API Keys & AWS Credentials**

To use Llumo and access your private S3 buckets, you need to set up the appropriate API keys and credentials.

1.  **Llumo API Key**: You can get your key from the [Llumo Dashboard](https://llumo.ai/dashboard).
2.  **AWS Credentials**: Required if you are accessing a **private** S3 bucket. If you are using the public sample bucket in this notebook, you can leave these blank.

In [ ]:
# Set your Llumo API Key
os.environ["LLUMO_API_KEY"] = getpass.getpass("Enter Your Llumo API Key: ")
llumo_key = os.getenv("LLUMO_API_KEY")

# ⚙️ Optional: Set AWS credentials for private S3 buckets
# If using a public bucket, you can skip these.
aws_access_key = getpass.getpass("Enter Your AWS Access Key ID (optional): ")
aws_secret_key = getpass.getpass("Enter Your AWS Secret Access Key (optional): ")

if aws_access_key:
    os.environ["AWS_ACCESS_KEY_ID"] = aws_access_key
if aws_secret_key:
    os.environ["AWS_SECRET_ACCESS_KEY"] = aws_secret_key

### **☁️ 4. Read Data from an AWS S3 Bucket**

This step connects to AWS S3 to fetch your dataset. We'll use a sample dataset from a public Llumo S3 bucket, which is in JSON Lines (`.jsonl`) format. 

**You can replace `bucket_name` and `file_key` with your own S3 bucket details.**

In [ ]:
# Define the S3 bucket and file path
# ➡️ TODO: Change these to your bucket name and file path if not using the sample
bucket_name = 'llumo-public' # Example: 'my-evaluation-logs-bucket'
file_key = 'datasets/sample_eval_data.jsonl' # Example: 'logs/june/batch-1.jsonl'

# Initialize Boto3 S3 client
s3_client_args = {}
# If you are using a public bucket that doesn't require signing, you can use the following:
if not aws_access_key:
    from botocore import UNSIGNED
    from botocore.config import Config
    s3_client_args['config'] = Config(signature_version=UNSIGNED)
    print("Connecting to S3 in anonymous mode for public bucket access.")
else:
    print("Connecting to S3 using provided AWS credentials.")

s3 = boto3.client('s3', **s3_client_args)

# Fetch the file from S3 and load the data
raw_logs = []
try:
    s3_object = s3.get_object(Bucket=bucket_name, Key=file_key)
    # Read the file line by line, assuming JSONL format
    for line in s3_object['Body'].iter_lines():
        raw_logs.append(json.loads(line.decode('utf-8')))
    print(f"Successfully loaded {len(raw_logs)} records from s3://{bucket_name}/{file_key}")
except Exception as e:
    print(f"Error loading data from S3: {e}")
    print("Please ensure your bucket name, file key, and AWS credentials (if required) are correct.")

# Preview the first raw log
if raw_logs:
    print("\nSample raw log from S3:")
    print(raw_logs[0])

### **🔄 5. Format Data for Llumo Evaluation**
Llumo's `evaluateMultiple` function expects a list of dictionaries, where each dictionary contains specific keys like `query`, `context`, and `output`. 

The function below converts our raw logs from S3 into this standardized format. You'll need to **adjust the key mappings** inside the function to match the structure of your own log files.

In [ ]:
def convert_to_llumo_format(logs):
  """
  Converts a list of raw log dictionaries into the format required by Llumo.
  
  Args:
    logs (list): A list of dictionaries, where each dictionary is a raw log.
    
  Returns:
    list: A list of formatted dictionaries for Llumo evaluation.
  """
  formatted_data = []
  for log in logs:
    # ➡️ TODO: Adjust these key names to match your data structure.
    # For our sample data, the keys are 'user_prompt', 'retrieved_documents', and 'llm_answer'.
    formatted_dict = {
        'query': log.get('user_prompt', ''),       # Map your user's question/prompt here
        'context': log.get('retrieved_documents', ''), # Map the retrieved context here
        'output': log.get('llm_answer', ''),        # Map the model's generated response here
        # Optional: You can also include ground truth if you have it
        'ground_truth': log.get('reference_answer', None) 
    }
    formatted_data.append(formatted_dict)
  return formatted_data

# Process the loaded logs
if raw_logs:
    evaluation_data = convert_to_llumo_format(raw_logs)
    
    # Preview the first formatted item to verify
    print("Sample log after formatting for Llumo:")
    print(json.dumps(evaluation_data[0], indent=2))
else:
    evaluation_data = []
    print("No data to format.")

### **🤖 6. Initialize Llumo Client and Evaluate Responses**

Now we're ready for the evaluation. We will initialize the `LlumoClient` and call the `evaluateMultiple` function.

We pass our formatted data and select the KPIs we want to measure:
- 🎯 **Response Correctness**: Is the answer factually accurate based on the context?
- 🧩 **Response Completeness**: Does the answer fully address the user's query?
- 🧠 **Response Bias**: Is the response free from demographic or social biases?
- ☣️ **Harmfulness**: Does the response contain toxic, hateful, or unsafe content?
- 🛠️ **Context Utilization**: How well does the answer use the provided context?
- ▶ **Hallucination**: Does the answer invent information not present in the context?

In [ ]:
evalDf = pd.DataFrame()

if evaluation_data and llumo_key:
    # Initialize the LlumoClient with your API key
    client = LlumoClient(api_key = llumo_key)

    # Call the evaluation function
    evalDf = client.evaluateMultiple(
      data = evaluation_data,  # The formatted data from the previous step
      evals = ["Response Completeness", "Response Correctness", "Response Bias", "Context Utilization", "Hallucination"], # Selected evaluation KPIs
      getDataFrame = True # Return result as a pandas DataFrame
    )
    print("Evaluation complete!")
else:
    print("Skipping evaluation. Ensure data was loaded and Llumo API key is set.")

### **📊 7. View Evaluation Results**
The results are returned in a pandas DataFrame, giving a detailed breakdown of each metric for every data point. This allows for easy analysis, sorting, and filtering to identify problematic responses.

In [ ]:
# Display the full evaluation results table
if not evalDf.empty:
    # To better view all columns
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', 80)
    display(evalDf)
else:
    print("No evaluation results to display.")